In [3]:
import pandas as pd

splits = {'train': 'plain_text/train-00000-of-00001.parquet', 'validation': 'plain_text/validation-00000-of-00001.parquet'}
df_squad_train = pd.read_parquet("hf://datasets/rajpurkar/squad/" + splits["train"])
df_squad_test = pd.read_parquet("hf://datasets/rajpurkar/squad/" + splits["validation"])

In [48]:
print(df_squad_train.columns)


print(df_squad_test)

Index(['id', 'title', 'context', 'question', 'answers'], dtype='object')
                             id          title  \
0      56be4db0acb8001400a502ec  Super_Bowl_50   
1      56be4db0acb8001400a502ed  Super_Bowl_50   
2      56be4db0acb8001400a502ee  Super_Bowl_50   
3      56be4db0acb8001400a502ef  Super_Bowl_50   
4      56be4db0acb8001400a502f0  Super_Bowl_50   
...                         ...            ...   
10565  5737aafd1c456719005744fb          Force   
10566  5737aafd1c456719005744fc          Force   
10567  5737aafd1c456719005744fd          Force   
10568  5737aafd1c456719005744fe          Force   
10569  5737aafd1c456719005744ff          Force   

                                                 context  \
0      Super Bowl 50 was an American football game to...   
1      Super Bowl 50 was an American football game to...   
2      Super Bowl 50 was an American football game to...   
3      Super Bowl 50 was an American football game to...   
4      Super Bowl 50 was a

In [5]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df_sciq_train = pd.read_parquet("hf://datasets/allenai/sciq/" + splits["train"])
df_sciq_test = pd.read_parquet("hf://datasets/allenai/sciq/" + splits["validation"])


In [6]:
print(df_sciq_train)
print(df_sciq_test)

                                                question      distractor3  \
0      What type of organism is commonly used in prep...          viruses   
1      What phenomenon makes global winds blow northe...  tropical effect   
2      Changes from a less-ordered state to a more-or...      endothermic   
3         What is the least dangerous radioactive decay?       zeta decay   
4      Kilauea in hawaii is the world’s most continuo...            magma   
...                                                  ...              ...   
11674  The enzyme pepsin plays an important role in t...           lipids   
11675  What remains a constant of radioactive substan...          acidity   
11676  Terrestrial ecosystems, also known for their d...       substrates   
11677  High explosives create shock waves that exceed...       turbulence   
11678  What do you call a structure composed of two o...           system   

            distractor1         distractor2        correct_answer  \
0     

In [7]:
import pandas as pd

splits = {'train': 'main/train-00000-of-00001.parquet', 'validation': 'main/validation-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}
df_open_book_train = pd.read_parquet("hf://datasets/allenai/openbookqa/" + splits["train"])
df_open_book_test = pd.read_parquet("hf://datasets/allenai/openbookqa/" + splits["validation"])


In [44]:
# print(df_open_book_train)
# print(df_open_book_test)
print(df_open_book_train)
print(df_open_book_test.shape)

           id                                      question_stem  \
0       7-980                         The sun is responsible for   
1       7-584       When standing miles away from Mount Rushmore   
2       7-870                When food is reduced in the stomach   
3       7-321                                          Stars are   
4       9-732                    You can make a telescope with a   
...       ...                                                ...   
4952  14-1506                     A bulldozer alters the area of   
4953  14-1509  An organism that can survive without the help ...   
4954  14-1510  The nimbleness of this animal is a key adaptio...   
4955  14-1511  Birds will have different kinds of beaks depen...   
4956  14-1512  Harriet wants to know the area of a rectangula...   

                                                choices answerKey  
0     {'text': ['puppies learning new tricks', 'chil...         D  
1     {'text': ['the mountains seem very close'

In [9]:
import json

def format_educational_data(df_squad, df_sciq, df_obqa):
    unified_data = []

    # 1. Process SQuAD (Grounded Question Answering)
    # We take the first answer from the 'answers' dictionary column
    for _, row in df_squad.head(3000).iterrows():
        unified_data.append({
            "instruction": "Based on the provided context, answer the student's question accurately.",
            "input": f"Context: {row['context']}\nQuestion: {row['question']}",
            "output": row['answers']['text'][0]
        })

    # 2. Process SciQ (Distractor & MCQ Generation)
    for _, row in df_sciq.head(3000).iterrows():
        # SciQ has distractors and support text
        context = row['support'] if row['support'] else "General Science Knowledge"
        unified_data.append({
            "instruction": "Generate a multiple-choice question with 3 plausible distractors based on this concept.",
            "input": f"Concept: {context}\nTopic: {row['question']}",
            "output": f"Correct Answer: {row['correct_answer']}\nDistractors: {row['distractor1']}, {row['distractor2']}, {row['distractor3']}"
        })

    # 3. Process OpenBookQA (Scientific Reasoning)
    # OpenBookQA has 'choices' as a dict and 'answerKey' as a letter (A, B, C, D)
    for _, row in df_obqa.head(3000).iterrows():
        choices_text = [f"{label}: {text}" for label, text in zip(row['choices']['label'], row['choices']['text'])]
        unified_data.append({
            "instruction": "Analyze the question and identify the correct scientific conclusion from the choices.",
            "input": f"Question: {row['question_stem']}\nChoices: {', '.join(choices_text)}",
            "output": f"The correct answer is {row['answerKey']}."
        })

    return unified_data

# Execute formatting
phase1_list = format_educational_data(df_squad_train, df_sciq_train, df_open_book_train)

In [10]:
output_file = "tutor_train_phase1.jsonl"

with open(output_file, 'w') as f:
    for entry in phase1_list:
        f.write(json.dumps(entry) + '\n')

print(f"✅ Phase 1 Complete! {len(phase1_list)} samples saved to {output_file}")

✅ Phase 1 Complete! 9000 samples saved to tutor_train_phase1.jsonl


In [11]:
from datasets import load_dataset

# Load the file we just created
dataset = load_dataset("json", data_files=output_file, split="train")

# Shuffle to mix SQuAD, SciQ, and OBQA together
dataset = dataset.shuffle(seed=42)

print(dataset)
# View a sample to verify
print("\nSample Data:")
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 9000
})

Sample Data:
{'instruction': "Based on the provided context, answer the student's question accurately.", 'input': "Context: The funeral, held at the Church of the Madeleine in Paris, was delayed almost two weeks, until 30 October. Entrance was restricted to ticket holders as many people were expected to attend. Over 3,000 people arrived without invitations, from as far as London, Berlin and Vienna, and were excluded.\nQuestion: Where was Chopin's funeral held?", 'output': 'Church of the Madeleine'}


In [25]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
print(torch.cuda.is_available())
max_seq_len = 2048
dtype = None
load_in_4bit = True
model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.2-3b-bnb-4bit",
    max_seq_length = max_seq_len,
dtype = dtype,
load_in_4bit = load_in_4bit
)



True
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-bnb-4bit as a legacy tokenizer.


In [27]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: Higher = more capacity but more VRAM. 16 is ideal for your study.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 for Unsloth
    bias = "none",    # Optimized to "none" for Unsloth
    # [UNSLOTH] Use gradient checkpointing to fit 2x more context
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,  
    loftq_config = None, 
)

# Research Metric: Check how many parameters we are actually training
trainable_params, all_param = model.get_nb_trainable_parameters()
print(f"Trainable params: {trainable_params:,d} || All params: {all_param:,d} || Percentage: {100 * trainable_params / all_param:.4f}%")

Unsloth 2026.3.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable params: 24,313,856 || All params: 3,237,063,680 || Percentage: 0.7511%


In [28]:
from datasets import load_dataset

# 1. Load the unified dataset you created in Phase 1
dataset = load_dataset("json", data_files="tutor_train_phase1.jsonl", split="train")

# 2. Define the "Teacher-Persona" Prompt Template
# This format ensures the model learns to follow specific educational instructions
tutor_prompt = """### Instruction:
{}

### Context/Input:
{}

### Teacher Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Essential: Tells the model when to stop talking

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # We combine the parts into the template and add the EOS_TOKEN
        text = tutor_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# 3. Apply the mapping (this is fast)
dataset = dataset.map(formatting_prompts_func, batched = True)

# 4. Verify the first sample
print("--- Formatted Sample ---")
print(dataset[0]["text"])

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

--- Formatted Sample ---
### Instruction:
Based on the provided context, answer the student's question accurately.

### Context/Input:
Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

### Teacher Response:
Saint Bernadette Soubirous<|end_of_text|>


In [33]:
import os
# Force Unsloth to use the standard training loop to avoid the .mean() error
os.environ["UNSLOTH_USE_COMPILED_TRAINER"] = "0"

In [38]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import os

# 1. Disable the compiled trainer just in case
os.environ["UNSLOTH_USE_COMPILED_TRAINER"] = "0"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_len,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, 
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        # --- THE CRITICAL FIX FOR 2026 ---
        average_tokens_across_devices = False, 
    ),
)

# --- START TRAINING ---
# This should now bypass the 'int' object error and start the progress bar!
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,2.580626
2,2.511665
3,2.661257
4,2.568092
5,2.667676
6,2.332543
7,2.442863
8,2.218991
9,1.848138
10,2.070702


In [42]:
from unsloth import FastLanguageModel

# 1. Set to Inference Mode
FastLanguageModel.for_inference(model)

# 2. Define a Test Case (Topic: Quantum Physics - something complex)
test_context = "Quantum entanglement is a phenomenon where two particles become linked, and the state of one instantly influences the state of the other, regardless of distance."
test_instruction = "As a helpful tutor, explain this concept and then generate one Multiple Choice Question with 3 distractors."

# 3. Generate the response
inputs = tokenizer(
    [tutor_prompt.format(test_instruction, test_context, "")], 
    return_tensors = "pt"
).to("cuda")

# Use a lower temperature (0.3) for factual teaching, higher (0.7) for creative quizzes
# Updated Inference Call
outputs = model.generate(
    **inputs, 
    max_new_tokens = 200, 
    temperature = 0.8,         # Higher = more creative/diverse
    repetition_penalty = 1.2,   # Punishes the model for repeating words
    top_p = 0.9,                # Only considers the top 90% of likely words
    do_sample = True            # Enables the temperature logic
)
response = tokenizer.decode(outputs[0], skip_special_tokens = True)

print("🎓 --- TUTOR EVALUATION ---")
print(response.split("### Teacher Response:")[1])

🎓 --- TUTOR EVALUATION ---

Correct Answer: quantum
Distractors: gravity, magnetism, friction


In [49]:
import os
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

# ==========================================
# 1. RESEARCH CONFIGURATION
# ==========================================
# Change to: "unsloth/Llama-3.2-1B-bnb-4bit" or "unsloth/Llama-3.1-8B-bnb-4bit" for other runs
MODEL_NAME = "unsloth/Llama-3.2-3B-bnb-4bit" 
ADAPTER_SAVE_NAME = "adapter_3b_squad_only"

# Assuming your pandas df is named 'df_squad'
# Standard SQuAD columns: 'context', 'question', 'answer' (or 'text')
# ==========================================

# 2. Convert Pandas to Hugging Face Dataset
dataset = Dataset.from_pandas(df_squad_train)

# 3. Load Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 4. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 5. Define Tutor Prompt & Formatting
tutor_prompt = """### Instruction:
Based on the provided context, answer the student's question accurately.

### Context:
{}

### Question:
{}

### Teacher Response:
{}"""

EOS_TOKEN = tokenizer.eos_token 

def formatting_prompts_func(examples):
    contexts  = examples["context"]
    questions = examples["question"]
    # 1. Use 'answers' (plural) to match your DataFrame
    # 2. Extract the first 'text' entry for the training label
    answers   = [a["text"][0] if len(a["text"]) > 0 else "" for a in examples["answers"]]
    
    texts = []
    for c, q, a in zip(contexts, questions, answers):
        text = tutor_prompt.format(c, q, a) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Now run the map again - it should work perfectly!
dataset = dataset.map(formatting_prompts_func, batched = True)
# 6. Training Execution (With 2026 Bug Fixes)
os.environ["UNSLOTH_USE_COMPILED_TRAINER"] = "0" 

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 100, 
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
        average_tokens_across_devices = False,
        seed = 3407,
    ),
)

trainer.train()

# 7. Save the SQuAD-Specific Adapter
model.save_pretrained(ADAPTER_SAVE_NAME)
print(f"✅ SQuAD Adapter saved as: {ADAPTER_SAVE_NAME}")

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/87599 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 87,599 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,2.398767
2,2.384271
3,2.284501
4,2.218736
5,2.118426
6,2.078961
7,2.176207
8,2.062982
9,1.930666
10,1.965114


✅ SQuAD Adapter saved as: adapter_3b_squad_only


In [50]:
from unsloth import FastLanguageModel
import torch

# 1. Switch to Fast Inference Mode
FastLanguageModel.for_inference(model) 

# 2. Define a Test Case (A concept from a different domain to test general logic)
test_context = """
The James Webb Space Telescope (JWST) is a space telescope designed primarily to conduct infrared astronomy. 
As the largest optical telescope in space, its high infrared resolution and sensitivity allow it to view 
objects too early, distant, or obscured for the Hubble Space Telescope.
"""
test_question = "What specifically allows the JWST to see objects that Hubble cannot?"

# 3. Format the Prompt (Must match your Training Template exactly!)
prompt = f"""### Instruction:
Based on the provided context, answer the student's question accurately.

### Context:
{test_context}

### Question:
{test_question}

### Teacher Response:
"""

# 4. Generate Answer
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs, 
    max_new_tokens = 50,
    temperature = 0.1,         # Low temp = more factual/less creative
    repetition_penalty = 1.2,
    use_cache = True
)

response = tokenizer.decode(outputs[0], skip_special_tokens = True)

# 5. Clean Output
print("🔬 --- ADAPTER TEST RESULT ---")
print(response.split("### Teacher Response:")[1].strip())

🔬 --- ADAPTER TEST RESULT ---
its high infrared resolution


In [52]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# --- SCENARIO 1: Technical Extraction (Science) ---
context_1 = "The Doppler effect is the change in frequency of a wave in relation to an observer who is moving relative to the wave source."
q_1 = "What determines the change in frequency in the Doppler effect?"

# --- SCENARIO 2: Reading Comprehension (History) ---
context_2 = "The Magna Carta was signed in June 1215 at Runnymede. It was a charter of rights agreed to by King John of England to make peace between the unpopular King and a group of rebel barons."
q_2 = "Who were the two main parties involved in the signing of the Magna Carta?"

# --- SCENARIO 3: Negative Constraint (The 'Hallucination' Test) ---
# This tests if the model is 'Grounded' or if it makes things up.
context_3 = "The company's new policy allows employees to work from home three days a week starting in July."
q_3 = "What is the name of the CEO of the company?" # Answer is NOT in text.

test_cases = [(context_1, q_1), (context_2, q_2), (context_3, q_3)]

print("🔬 --- COMPREHENSIVE ADAPTER BENCHMARK ---")

for i, (ctx, q) in enumerate(test_cases):
    prompt = f"### Instruction:\nBased on the provided context, answer the student's question accurately and explain the answer in detail.\n\n### Context:\n{ctx}\n\n### Question:\n{q}\n\n### Teacher Response:\n"
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens = 50, temperature = 0.1, repetition_penalty = 1.2)
    response = tokenizer.decode(outputs[0], skip_special_tokens = True).split("### Teacher Response:")[1].strip()
    
    print(f"\nTEST {i+1}:")
    print(f"❓ Q: {q}")
    print(f"🤖 A: {response}")
    print("-" * 30)

🔬 --- COMPREHENSIVE ADAPTER BENCHMARK ---

TEST 1:
❓ Q: What determines the change in frequency in the Doppler effect?
🤖 A: the observer
------------------------------

TEST 2:
❓ Q: Who were the two main parties involved in the signing of the Magna Carta?
🤖 A: King John of England and a group of rebel barons
------------------------------

TEST 3:
❓ Q: What is the name of the CEO of the company?
🤖 A: Satya Nadella
------------------------------
